# Verifying Amplitudes and Correlation Energies

TODO:
- Create a pandas dataframe with the following structure:

structure | basis set | PySCF_Corr | Psi4_Corr | PySCF_norm_t1 | Psi4_norm_t1 | PySCF_norm_t2 | Psi4_norm_t2

- Populate the data frame with the provided code
- Use a try-except to handle linalg error, save a list of faulty structures
- Calculate the mean error for each structure basis combo


In [45]:
import psutil
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm


from glob import glob
import psi4
from helper_CC_ML_spacial import *

import pyscf
import pyscf.cc
import pyscf.mcscf


In [46]:
import pandas as pd
import numpy as np
import glob
import os
from numpy.linalg import LinAlgError
import psi4
import pyscf
from pyscf import gto, scf, cc

# Import HelperCCEnergy for Psi4

# Path to XYZ files
xyz_folder = 'diatomics'
xyz_files = glob.glob(os.path.join(xyz_folder, '*.xyz'))

# Extract structure names and contents
structures = []
structure_contents = {}

for file_path in xyz_files:
    structure_name = os.path.splitext(os.path.basename(file_path))[0]
    structures.append(structure_name)
    with open(file_path, 'r') as f:
        structure_contents[structure_name] = f.read()

# Basis sets to iterate over

basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

results = []
failures = []

for structure in structures:
    xyz_path = os.path.join(xyz_folder, f'{structure}.xyz')
    with open(xyz_path, 'r') as f:
        xyz_text = f.read()

    for basis in basis_sets:
        # Initialize default values
        psi4_corr = np.nan
        pyscf_corr = np.nan
        norm_t1_psi4 = np.nan
        norm_t2_psi4 = np.nan
        norm_t1_pyscf = np.nan
        norm_t2_pyscf = np.nan

        ##### PSI4 Processing #####
        try:
            qmol = psi4.qcdb.Molecule.from_string(xyz_text, dtype='xyz')
            mol_psi4 = psi4.geometry(qmol.create_psi4_string_from_molecule() + 'symmetry c1')

            psi4.core.clean()
            psi4.core.be_quiet()

            psi4.set_options({
                'basis': basis,
                'scf_type': 'pk',
                'reference': 'rhf',
                'mp2_type': 'conv',
                'e_convergence': 1e-8,
                'd_convergence': 1e-8
            })

            rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            A = HelperCCEnergy(mol_psi4, rhf_e, scf_wfn, freeze_core=False)

            try:
                A.compute_energy()
                psi4_corr = A.FinalEnergy
            except LinAlgError:
                failures.append({'structure': structure, 'basis set': basis, 'method': 'Psi4_Corr'})

            # Extract amplitudes
            norm_t1_psi4 = np.linalg.norm(A.t1)
            norm_t2_psi4 = np.linalg.norm(A.t1)

        except Exception as e:
            failures.append({'structure': structure, 'basis set': basis, 'method': 'Psi4_Setup', 'error': str(e)})

        ##### PYSCF Processing #####
        try:
            mol_pyscf = gto.Mole()
            mol_pyscf.build(atom=xyz_path, basis=basis, symmetry='c1')

            # Define active space (assuming no frozen orbitals for simplicity)
            n_frozen = 0
            active_space = range(n_frozen, mol_pyscf.nao_nr())
            scf_calc = scf.RHF(mol_pyscf).run()
            

            # Compute CCSD
            ccsd_calc = cc.CCSD(scf_calc, frozen=[i for i in range(mol_pyscf.nao_nr()) if i not in active_space]).run()
            pyscf_corr = ccsd_calc.e_corr

            t1_pyscf = ccsd_calc.t1
            t2_pyscf = ccsd_calc.t2
            norm_t1_pyscf = np.linalg.norm(t1_pyscf)
            norm_t2_pyscf = np.linalg.norm(t2_pyscf)

        except LinAlgError:
            failures.append({'structure': structure, 'basis set': basis, 'method': 'PySCF_Corr'})
        except Exception as e:
            failures.append({'structure': structure, 'basis set': basis, 'method': 'PySCF_Setup', 'error': str(e)})

        # Store results
        results.append({
            'structure': structure,
            'basis set': basis,
            'PySCF_Corr': pyscf_corr,
            'Psi4_Corr': psi4_corr,
            'PySCF_norm_t1': norm_t1_pyscf,
            'Psi4_norm_t1': norm_t1_psi4,
            'PySCF_norm_t2': norm_t2_pyscf,
            'Psi4_norm_t2': norm_t2_psi4
        })

# Create and display final DataFrame
df = pd.DataFrame(results)
print("\nFinal DataFrame:")
print(df)

if failures:
    print("\nFailures Encountered:")
    for failure in failures:
        print(f"Structure: {failure['structure']}, Basis Set: {failure['basis set']}, Method: {failure['method']}, Error: {failure.get('error', '')}")

# Optional: Save results to CSV
# df.to_csv('amplitude_correlation_results.csv', index=False)


Computing RHF reference.


/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 6 basis functions.
(6, 6)
(6, 6)
Building initial guess...

..initialized CCSD in 0.013 seconds.

CCSD Iteration   0: CCSD correlation = -0.026326310201129   dE =  2.63263E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.036474688643062   dE = -1.01484E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.040882897046905   dE = -4.40821E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.044820008618742   dE = -3.93711E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.045713492152981   dE = -8.93484E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.046573303575851   dE = -8.59811E-04   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.050046718490928   dE = -3.47341E-03   DIIS = 5
CCSD Iteration   7: CCSD correlation = -0.070574298008330   dE = -2.05276E-02   DIIS = 6
CCSD Iteration   8: CCSD correlation = -0.056069359416175   dE =  1.45049E-02   DIIS = 7
CCSD 

/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:552: RuntimeWarning: divide by zero encountered in log10
  self.t2mag=np.log10(np.absolute(self.t2))
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:593: RuntimeWarning: divide by zero encountered in log10
  self.Jia1mag=np.log10(np.absolute(self.Jia1))
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:594: RuntimeWarning: divide by zero encountered in log10
  self.Jia2mag=np.log10(np.absolute(self.Jia2))
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:595: RuntimeWarning: divide by zero encountered in log10
  self.Kia1mag=np.log10(np.absolute(self.Kia1))
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:596: RuntimeWarning: divide by zero encountered in log10
  self.Kia2mag=np.log10(np.absolute(self.Kia2))


converged SCF energy = -54.1350442502145
E(CCSD) = -54.19702330469029  E_corr = -0.06197905447580512
Computing RHF reference.


...


Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 10 basis functions.
(10, 10)
(10, 10)
Building initial guess...

..initialized CCSD in 0.030 seconds.

CCSD Iteration   0: CCSD correlation = -0.213294471719363   dE =  2.13294E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.163296898936199   dE =  4.99976E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.206869295067322   dE = -4.35724E-02   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.196401421636421   dE =  1.04679E-02   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.207106089866467   dE = -1.07047E-02   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.211409691153762   dE = -4.30360E-03   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.211309901307862   dE =  9.97898E-05   DIIS = 5
CCSD Iteration   7: CCSD correlation = -0.211812201477656   dE = -5.02300E-04   DIIS = 6
CCSD Iteration   8: CCSD correlation = -0.211890004153050   dE = -7.78027E-05   DIIS = 7


/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 28 basis functions.
(28, 28)
(28, 28)
Building initial guess...

..initialized CCSD in 0.488 seconds.

CCSD Iteration   0: CCSD correlation = -0.294619018151936   dE =  2.94619E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.250382839574891   dE =  4.42362E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.290160710381002   dE = -3.97779E-02   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.280061011382206   dE =  1.00997E-02   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.284768190988494   dE = -4.70718E-03   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.289699576317932   dE = -4.93139E-03   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.290283973348879   dE = -5.84397E-04   DIIS = 5
CCSD Iteration   7: CCSD correlation = -0.290486431086653   dE = -2.02458E-04   DIIS = 6
CCSD Iteration   8: CCSD correlation = -0.290564652109172   dE = -7.82210E-05   DIIS = 7


/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 46 basis functions.
(46, 46)
(46, 46)
Building initial guess...

..initialized CCSD in 1.143 seconds.

CCSD Iteration   0: CCSD correlation = -0.301731793457221   dE =  3.01732E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.257685818595398   dE =  4.40460E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.296578253009507   dE = -3.88924E-02   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.286631626820871   dE =  9.94663E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.291046067513962   dE = -4.41444E-03   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.296004834244568   dE = -4.95877E-03   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.296572549777457   dE = -5.67716E-04   DIIS = 5
CCSD Iteration   7: CCSD correlation = -0.296825803820292   dE = -2.53254E-04   DIIS = 6
CCSD Iteration   8: CCSD correlation = -0.296897183486463   dE = -7.13797E-05   DIIS = 7


In [47]:
failures_df =pd.DataFrame(failures)

In [48]:
df

,structure,basis set,PySCF_Corr,Psi4_Corr,PySCF_norm_t1,Psi4_norm_t1,PySCF_norm_t2,Psi4_norm_t2
0,CN,STO-3G,NaN,NaN,NaN,NaN,NaN,NaN
1,CN,cc-pVDZ,NaN,NaN,NaN,NaN,NaN,NaN
2,CN,aug-cc-pVDZ,NaN,NaN,NaN,NaN,NaN,NaN
3,HN,STO-3G,-0.061979,-0.061979,0.008507,0.008507,1.006604,0.008507
4,HN,cc-pVDZ,-0.153711,-0.153711,0.015478,0.015479,0.426481,0.015479
...,...,...,...,...,...,...,...,...
103,HO,cc-pVDZ,NaN,NaN,NaN,NaN,NaN,NaN
104,HO,aug-cc-pVDZ,NaN,NaN,NaN,NaN,NaN,NaN
105,BeH,STO-3G,NaN,NaN,NaN,NaN,NaN,NaN
106,BeH,cc-pVDZ,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
df.to_csv('results.csv')


In [50]:
df.loc[df['structure'] == 'BB']

,structure,basis set,PySCF_Corr,Psi4_Corr,PySCF_norm_t1,Psi4_norm_t1,PySCF_norm_t2,Psi4_norm_t2
66,BB,STO-3G,-0.183195,-0.183195,0.124924,0.124924,0.582370,0.124924
67,BB,cc-pVDZ,-0.193752,-0.193752,0.096490,0.096490,0.516084,0.096490
68,BB,aug-cc-pVDZ,-0.195948,-0.195948,0.098257,0.098256,0.517198,0.098256
